# Intent dataset generation (S5.6/S5.8)

Generates intents across the coverage matrix and chains every one through schema validation, intent lint, expansion, spec lint and compilation. Fixed seed; failures are categorised, never hidden.

In [ ]:
# Havyn Stage 5 - environment setup and verification gate.
# Every notebook runs this first: no notebook trains or generates data on an
# unverified environment.
import sys, shutil, os
if not shutil.which("git"):
    raise RuntimeError("git missing")
!git clone --depth 1 https://github.com/HamzaNinjaCoder1/MCP-Packet-Tracer.git /content/MCP-Packet-Tracer 2>/dev/null || git -C /content/MCP-Packet-Tracer pull --ff-only
import sys
sys.path.insert(0, "/content/MCP-Packet-Tracer/havyn-stage5/scripts")
os.chdir("/content/MCP-Packet-Tracer")
from colab_setup import setup, save_report
env = setup(run_tests=True, use_drive=True)
assert not env.errors, f"environment verification failed: {env.errors}"
assert env.tests_passed, "offline suite failed - STOP, do not continue"
from env_report import build_report, recommend_config
report = build_report(repo="/content/MCP-Packet-Tracer", dataset_version="v0")
import json, pathlib
out_dir = pathlib.Path(env.drive_dir or "/content/havyn-artifacts")
out_dir.mkdir(parents=True, exist_ok=True)
(out_dir / "env-report.json").write_text(json.dumps(report, indent=2))
print(json.dumps(recommend_config(report.get("gpu", {})), indent=2))
print(json.dumps(report, indent=2)[:2000])


In [ ]:
import os
os.environ["HAVYN_DRIVE"] = env.drive_dir or "/content/havyn-artifacts"
os.environ["HAVYN_REPO"] = "/content/MCP-Packet-Tracer"
print("HAVYN_DRIVE =", os.environ["HAVYN_DRIVE"])

In [ ]:
from colab_ux import dry_run
if dry_run():
    print("[DRY_RUN] data generation skipped")
else:
    !PYTHONPATH=. python havyn-stage5/scripts/generate_intents.py --out havyn-stage5/datasets/havyn-intents-v1

In [ ]:
from colab_ux import dry_run
if dry_run():
    print("[DRY_RUN] prompt generation skipped")
else:
    # S5.9 template share: deterministic, no GPU. Teacher share (--teacher)
    # needs a GPU runtime and lands in REVIEW until round-trip.
    !PYTHONPATH=havyn-stage5/scripts:. python havyn-stage5/scripts/generate_prompts.py --per-cell 4

In [ ]:
import json
cov = json.load(open('havyn-stage5/datasets/havyn-intents-v1/coverage-report.json'))
print(json.dumps(cov, indent=2)[:3000])